# Wellness Centre ERPNext Migration - Validation & Reporting

**Version:** 1.0
**Purpose:** Prove migration completeness through CSV vs ERPNext reconciliation
**Prerequisite:** Migration notebook complete (all 6 phases run successfully)

# Phase A: Environment Setup

In [37]:
import sys
import importlib
from pathlib import Path
from datetime import datetime
import pandas as pd
import csv
import os
import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR   = REPO_ROOT / 'src'
DATA_DIR  = REPO_ROOT / 'data' / 'wellness_centre'
DOCS_DIR  = REPO_ROOT / 'docs'
REPORTS_DIR = REPO_ROOT / 'reports'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DOCS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

print(f"Repo:    {REPO_ROOT}")
print(f"Source:  {SRC_DIR}")
print(f"Data:    {DATA_DIR}")
print(f"Docs:    {DOCS_DIR}")
print(f"Reports: {REPORTS_DIR}")

Repo:    /home/jovyan/work/ERP/emt
Source:  /home/jovyan/work/ERP/emt/src
Data:    /home/jovyan/work/ERP/emt/data/wellness_centre
Docs:    /home/jovyan/work/ERP/emt/docs
Reports: /home/jovyan/work/ERP/emt/reports


In [38]:
from frappeclient import FrappeClient

# ── Edit these ──────────────────────────────────────────────────────────
ERPNEXT_URL    = "http://erpnext-frontend:8080"
ERPNEXT_DOMAIN = "well.rosslyn.cloud"
COMPANY        = "Wellness Centre"
API_KEYS_FILE  = "/home/jovyan/work/ERP/emt/config/api_keys.csv"
# ────────────────────────────────────────────────────────────────────────

API_KEY = API_SECRET = ""
if os.path.exists(API_KEYS_FILE):
    with open(API_KEYS_FILE, newline="") as f:
        for row in csv.DictReader(f):
            API_KEY    = row.get("api_key", "").strip()
            API_SECRET = row.get("api_secret", "").strip()
            break
    print(f"✓ Credentials loaded")
else:
    print(f"⚠ Keys file not found: {API_KEYS_FILE}")

client = FrappeClient(ERPNEXT_URL)
client.authenticate(API_KEY, API_SECRET)

# Host header for internal Docker routing
import requests
client.session.headers.update({"Host": ERPNEXT_DOMAIN})

# Quick connection test — list one document to confirm credentials work
test = client.get_list("Company", fields=["name"], limit_page_length=1)
print(f"✓ Connected. Company found: {test[0]['name']}")

✓ Credentials loaded
✓ Connected. Company found: Wellness Centre


# Phase B: ValidationReporter

In [45]:
# Force reload
import sys
if 'orchestration.validation_reporter' in sys.modules:
    del sys.modules['orchestration.validation_reporter']

from orchestration.validation_reporter import ValidationReporter
print(f"✓ v{ValidationReporter.VERSION}")

✓ v1.1


In [46]:
reporter = ValidationReporter(
    client=client,
    data_dir=DATA_DIR,
    company=COMPANY
)

report = reporter.run()

VALIDATION REPORTER v1.1

[1/3] Financial reconciliation...
[2/3] Inventory reconciliation...
[3/3] Operational reconciliation...

COMPLETE: 13/13 checks passed (100.0%)


In [47]:
# Save the markdown report
from pathlib import Path
reporter.save_markdown(report, Path("../docs/validation_report.md"))

✓ Report saved: ../docs/validation_report.md


PosixPath('../docs/validation_report.md')

In [50]:
# Reload
import sys
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['erpnext_fetcher','report_generator']):
        del sys.modules[mod]

from orchestration.report_generator import ReportGenerator

gen = ReportGenerator(client=client, data_dir=DATA_DIR, company=COMPANY)
gen.build("../docs/reports/Wellness_Centre_Reconciliation_v2.xlsx")

REPORT GENERATOR v2.0

[1/3] Loading source CSV data...
  ✓ 9 source datasets loaded
[2/3] Fetching live ERPNext data...
  ✓ etims: 220 records
  ✓ room_inv: 54 records
  ✓ event_inv: 25 records
  ✓ egg_inv: 103 records
  ✓ payments: 220 records
  ✓ journal: 727 records
  ✓ items: 77 records
  ✓ stock_entries: 190 records
  ✓ compliance: 9 records
[3/3] Building workbook...

✓ Saved: ../docs/reports/Wellness_Centre_Reconciliation_v2.xlsx
  Sheets: ['1. Summary', '2. Revenue Detail', '3. Expense Detail', '4. Inventory', '5. Poultry Farm', '6. Compliance', '7. Monthly P&L']


PosixPath('../docs/reports/Wellness_Centre_Reconciliation_v2.xlsx')